# Maximum Likelihood Estimation (MLE): Analytical vs. Numerical Calibration

## 1. Introduction and Theoretical Framework

In financial engineering, calibrating a theoretical probability distribution to empirical market data is a fundamental prerequisite for pricing, risk management, and algorithmic trading. Maximum Likelihood Estimation (MLE) is the most robust estimator due to its asymptotic properties: consistency, asymptotic normality, and efficiency (achieving the Cramér-Rao lower bound).

Let $X = (x_1, x_2, \dots, x_n)$ be a vector of independent and identically distributed (i.i.d.) observations drawn from a probability density function $f(x|\boldsymbol{\theta})$, where $\boldsymbol{\theta}$ represents the vector of unknown parameters. 

The likelihood function is defined as the joint density of the observations:
$$\mathcal{L}(\boldsymbol{\theta}|X) = \prod_{i=1}^{n} f(x_i|\boldsymbol{\theta})$$

For analytical tractability and numerical stability, we monotonically transform this using the natural logarithm to obtain the log-likelihood function, $\ell(\boldsymbol{\theta}|X)$:
$$\ell(\boldsymbol{\theta}|X) = \sum_{i=1}^{n} \ln f(x_i|\boldsymbol{\theta})$$

The MLE estimator $\hat{\boldsymbol{\theta}}$ is the argument that maximizes this function:
$$\hat{\boldsymbol{\theta}}_{MLE} = \arg\max_{\boldsymbol{\theta}} \ell(\boldsymbol{\theta}|X)$$

This optimization reduces to finding the roots of the score function, $S(\boldsymbol{\theta}) = \nabla \ell(\boldsymbol{\theta}|X) = 0$. Depending on the distribution, this system of equations either yields a **closed-form analytical solution** or necessitates **numerical algorithmic methods** such as the Newton-Raphson method.

## 2. Analytical Calibration (Closed-Form Solution)

When the score function can be solved algebraically for $\boldsymbol{\theta}$, we obtain a closed-form equation. The Gaussian (Normal) distribution is the canonical example, heavily utilized in Black-Scholes dynamics.

For $X \sim \mathcal{N}(\mu, \sigma^2)$, the log-likelihood is:
$$\ell(\mu, \sigma^2) = -\frac{n}{2}\ln(2\pi) - \frac{n}{2}\ln(\sigma^2) - \frac{1}{2\sigma^2}\sum_{i=1}^{n} (x_i - \mu)^2$$

Taking the partial derivatives with respect to $\mu$ and $\sigma^2$ and equating them to zero yields the classic closed-form estimators:
$$\hat{\mu} = \frac{1}{n}\sum_{i=1}^{n} x_i$$
$$\hat{\sigma}^2 = \frac{1}{n}\sum_{i=1}^{n} (x_i - \hat{\mu})^2$$

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import digamma, polygamma
import time

# Set random seed for reproducibility
np.random.seed(42)

def closed_form_gaussian_mle(data):
    """
    Computes the analytical MLE for a Gaussian distribution.
    """
    n = len(data)
    mu_hat = np.sum(data) / n
    sigma2_hat = np.sum((data - mu_hat)**2) / n
    return mu_hat, sigma2_hat

# Generate empirical data representing log-returns
true_mu, true_sigma = 0.05, 0.20
gaussian_data = np.random.normal(true_mu, true_sigma, 1000)

start_time = time.time()
mu_mle, sigma2_mle = closed_form_gaussian_mle(gaussian_data)
analytical_time = time.time() - start_time

print(f"--- Analytical Gaussian MLE ---")
print(f"Estimated Mu: {mu_mle:.4f} (True: {true_mu:.4f})")
print(f"Estimated Volatility (Sigma): {np.sqrt(sigma2_mle):.4f} (True: {true_sigma:.4f})")
print(f"Computation Time: {analytical_time:.6f} seconds\n")

--- Analytical Gaussian MLE ---
Estimated Mu: 0.0539 (True: 0.0500)
Estimated Volatility (Sigma): 0.1957 (True: 0.2000)
Computation Time: 0.000231 seconds



## 3. Algorithmic Calibration (Newton-Raphson Method)

Many distributions critical to quantitative finance (e.g., Gamma for Loss Given Default, Weibull for survival analysis) do not permit closed-form solutions for all parameters. Here, we must employ numerical optimization.

The Newton-Raphson algorithm (see [Newton Raphson.ipynb](Probability/0.%20Calculus/Newton%20Raphson.ipynb)) is a second-order root-finding method. To find the root of the score function $S(\boldsymbol{\theta}) = 0$, we utilize the iterative update rule:
$$\boldsymbol{\theta}_{k+1} = \boldsymbol{\theta}_k - [\mathcal{H}(\boldsymbol{\theta}_k)]^{-1} S(\boldsymbol{\theta}_k)$$
where $\mathcal{H}$ is the Hessian matrix (the second derivative of the log-likelihood).

### Example: Gamma Distribution Shape Parameter
Let $X \sim \text{Gamma}(\alpha, \beta)$ where $\alpha$ is the shape and $\beta$ is the rate. 
By substituting the analytical solution for the rate $\hat{\beta} = \alpha / \bar{x}$ into the log-likelihood, we obtain a concentrated log-likelihood depending solely on $\alpha$. The score function and Hessian for $\alpha$ become:

$$S(\alpha) = n\ln(\alpha) - n\ln(\bar{x}) - n\psi(\alpha) + \sum_{i=1}^{n} \ln(x_i)$$
$$\mathcal{H}(\alpha) = \frac{n}{\alpha} - n\psi_1(\alpha)$$

where $\psi(\alpha)$ is the digamma function and $\psi_1(\alpha)$ is the trigamma function. We iterate $\alpha_{k+1} = \alpha_k - S(\alpha_k)/\mathcal{H}(\alpha_k)$ until convergence.

In [2]:
def newton_raphson_gamma_mle(data, tol=1e-6, max_iter=100):
    """
    Computes the MLE for a Gamma distribution using the Newton-Raphson algorithm 
    for the shape parameter (alpha), and closed-form for the rate parameter (beta).
    """
    n = len(data)
    x_bar = np.mean(data)
    sum_log_x = np.sum(np.log(data))
    
    # Method of Moments estimator for initial guess
    var_x = np.var(data)
    alpha_k = (x_bar ** 2) / var_x 
    
    for i in range(max_iter):
        # Score function (First derivative of log-likelihood w.r.t alpha)
        score = n * np.log(alpha_k) - n * np.log(x_bar) - n * digamma(alpha_k) + sum_log_x
        
        # Hessian (Second derivative w.r.t alpha)
        # polygamma(1, x) is the trigamma function
        hessian = (n / alpha_k) - n * polygamma(1, alpha_k)
        
        # Newton-Raphson update
        alpha_next = alpha_k - (score / hessian)
        
        if abs(alpha_next - alpha_k) < tol:
            alpha_k = alpha_next
            break
            
        alpha_k = alpha_next
        
    # Calculate beta using the converged alpha
    beta_k = alpha_k / x_bar
    return alpha_k, beta_k, i+1

# Generate empirical data representing positive continuous values (e.g., LGD, durations)
true_alpha, true_beta = 2.5, 1.2
gamma_data = np.random.gamma(shape=true_alpha, scale=1/true_beta, size=1000)

start_time = time.time()
alpha_mle, beta_mle, iterations = newton_raphson_gamma_mle(gamma_data)
numerical_time = time.time() - start_time

print(f"--- Numerical Gamma MLE (Newton-Raphson) ---")
print(f"Estimated Shape (Alpha): {alpha_mle:.4f} (True: {true_alpha:.4f})")
print(f"Estimated Rate (Beta): {beta_mle:.4f} (True: {true_beta:.4f})")
print(f"Converged in {iterations} iterations.")
print(f"Computation Time: {numerical_time:.6f} seconds\n")

--- Numerical Gamma MLE (Newton-Raphson) ---
Estimated Shape (Alpha): 2.6987 (True: 2.5000)
Estimated Rate (Beta): 1.2654 (True: 1.2000)
Converged in 3 iterations.
Computation Time: 0.003573 seconds



## 4. Comparative Analysis

### Methodological Trade-offs
1. **Computational Complexity**: Closed-form solutions require $O(n)$ operations to compute sufficient statistics (mean, variance). Algorithmic methods require $O(k \cdot n)$ operations, where $k$ is the number of iterations to convergence. In our benchmark, the analytical method is strictly faster.
2. **Determinism**: Analytical equations yield exact deterministic results. Newton-Raphson is subject to floating-point precision constraints, requires a robust initial guess (we used the Method of Moments estimator), and carries the risk of diverging or converging to local optima if the log-likelihood is not strictly concave.
3. **Generality**: While analytical solutions are mathematically elegant, they are restricted to a narrow class of exponential family distributions. Algorithmic root-finding is universally applicable, forming the basis of training algorithms in modern Machine Learning (e.g., Gradient Descent is a first-order relative of Newton-Raphson).